# Assignment 05 · Notebook 02: Ba tập dữ liệu

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Notebook phục vụ **mục 3** của đề: hai tập ảnh và một tập về bệnh tiểu đường.

| Tập | Nguồn | Quy mô |
|---|---|---|
| CIFAR-10 | https://www.cs.toronto.edu/~kriz/cifar.html | 60 000 ảnh 32×32×3, 10 lớp |
| CIFAR-100 | https://www.cs.toronto.edu/~kriz/cifar.html | 60 000 ảnh 32×32×3, 100 lớp mịn, 20 siêu lớp |
| Diabetes Prediction | https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset | 100 000 dòng, 8 đặc trưng |

Đề ghi "MNIST-100"; không có bộ dữ liệu chuẩn nào mang tên này, nên bài hiểu đó là CIFAR-100,
tập 100 lớp cùng họ với CIFAR-10. Mọi hình của chương 3 được vẽ bằng pgfplots từ các tệp `.dat`
xuất ở đây; ảnh mẫu được ghi thành từng điểm ảnh RGB.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from a05.data import (DIAB_CATEGORICAL, DIAB_NUMERIC, DIAB_TARGET, load_cifar, load_diabetes_raw,
                      prepare_diabetes, split_train_val)
from a05.export import save_metrics, write_dat, write_image_dat, write_matrix_dat

print("Thiết bị: CPU (NumPy", np.__version__, "· pandas", pd.__version__ + ")")


def tile(images, rows, cols, gap=9):
    """Ghép các ảnh 32×32×3 thành một lưới, ngăn cách bởi viền trắng rộng `gap` điểm ảnh."""
    h, w = images[0].shape[:2]
    out = np.full((rows * h + (rows - 1) * gap, cols * w + (cols - 1) * gap, 3), 255, np.uint8)
    for k, im in enumerate(images):
        r, c = divmod(k, cols)
        out[r * (h + gap):r * (h + gap) + h, c * (w + gap):c * (w + gap) + w] = im
    return out


def pixel_hist(x, bins=32):
    edges = np.linspace(0, 256, bins + 1)
    cols = {"bin": (edges[:-1] + edges[1:]) / 2}
    for ch, name in enumerate("rgb"):
        h, _ = np.histogram(x[..., ch], bins=edges, density=True)
        cols[name] = h
    return cols


def describe_images(d, name):
    tr, va = split_train_val(d["y_train"])
    x = d["x_train"][tr] / 255
    counts = np.bincount(d["y_train"], minlength=len(d["class_names"]))
    info = {"n_train": len(tr), "n_val": len(va), "n_test": len(d["y_test"]),
            "n_classes": len(d["class_names"]), "per_class_min": counts.min(), "per_class_max": counts.max(),
            "test_per_class": np.bincount(d["y_test"]).min(),
            "mean": x.mean(axis=(0, 1, 2)), "std": x.std(axis=(0, 1, 2)), "shape": "×".join(map(str, d["x_train"].shape[1:]))}
    print(name, {k: (np.round(v, 4) if isinstance(v, np.ndarray) else v) for k, v in info.items()})
    write_dat(f"{name}_pixel_hist", pixel_hist(d["x_train"][tr]))
    return info

Thiết bị: CPU (NumPy 2.5.1 · pandas 3.0.5)


## CIFAR-10
Mỗi lớp lấy ảnh huấn luyện đầu tiên thuộc lớp đó, ghép thành lưới 2×5.

In [2]:
c10 = load_cifar("cifar10")
print({k: v.shape for k, v in c10.items() if isinstance(v, np.ndarray)})
print(c10["class_names"])
idx10 = [int(np.flatnonzero(c10["y_train"] == k)[0]) for k in range(10)]
write_image_dat("cifar10_samples", tile([c10["x_train"][i] for i in idx10], 2, 5))
info10 = describe_images(c10, "cifar10")
info10["class_names"] = c10["class_names"]

{'x_train': (50000, 32, 32, 3), 'y_train': (50000,), 'x_test': (10000, 32, 32, 3), 'y_test': (10000,)}
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


cifar10 {'n_train': 45000, 'n_val': 5000, 'n_test': 10000, 'n_classes': 10, 'per_class_min': np.int64(5000), 'per_class_max': np.int64(5000), 'test_per_class': np.int64(1000), 'mean': array([0.4911, 0.4821, 0.4464]), 'std': array([0.2469, 0.2434, 0.2616]), 'shape': '32×32×3'}


## CIFAR-100
100 lớp mịn được nhóm thành 20 siêu lớp; mỗi siêu lớp lấy một ảnh, ghép thành lưới 4×5.

In [3]:
c100 = load_cifar("cifar100")
print({k: v.shape for k, v in c100.items() if isinstance(v, np.ndarray)})
idx100 = [int(np.flatnonzero(c100["yc_train"] == k)[0]) for k in range(20)]
write_image_dat("cifar100_samples", tile([c100["x_train"][i] for i in idx100], 4, 5))
info100 = describe_images(c100, "cifar100")
info100["coarse_names"] = c100["coarse_names"]
info100["sample_fine"] = [c100["class_names"][c100["y_train"][i]] for i in idx100]
fine_per_coarse = np.bincount(np.unique(np.stack([c100["y_train"], c100["yc_train"]], 1), axis=0)[:, 1])
info100["fine_per_coarse"] = int(fine_per_coarse.min())
print("số lớp mịn mỗi siêu lớp:", set(fine_per_coarse.tolist()))

{'x_train': (50000, 32, 32, 3), 'y_train': (50000,), 'yc_train': (50000,), 'x_test': (10000, 32, 32, 3), 'y_test': (10000,), 'yc_test': (10000,)}


cifar100 {'n_train': 45000, 'n_val': 5000, 'n_test': 10000, 'n_classes': 100, 'per_class_min': np.int64(500), 'per_class_max': np.int64(500), 'test_per_class': np.int64(100), 'mean': array([0.5071, 0.4864, 0.4406]), 'std': array([0.2672, 0.2563, 0.276 ]), 'shape': '32×32×3'}


số lớp mịn mỗi siêu lớp: {5}


## Diabetes Prediction Dataset

In [4]:
raw = load_diabetes_raw()
print(raw.shape)
print(raw.head())
dup = int(raw.duplicated().sum())
df = raw.drop_duplicates()
desc = df[DIAB_NUMERIC].describe().T[["mean", "std", "min", "max"]]
print("Số dòng trùng:", dup)
print(desc.round(2))
print(df[DIAB_TARGET].value_counts())
print(df["smoking_history"].value_counts(normalize=True).round(4))

(100000, 9)
   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  
3          5.0                  155         0  
4          4.8                  155         0  


Số dòng trùng: 3854
                       mean    std    min     max
age                   41.79  22.46   0.08   80.00
bmi                   27.32   6.77  10.01   95.69
HbA1c_level            5.53   1.07   3.50    9.00
blood_glucose_level  138.22  40.91  80.00  300.00
diabetes
0    87664
1     8482
Name: count, dtype: int64
smoking_history
never          0.3578
No Info        0.3421
former         0.0967
current        0.0957
not current    0.0662
ever           0.0416
Name: proportion, dtype: float64


In [5]:
pos = df[df[DIAB_TARGET] == 1]
neg = df[df[DIAB_TARGET] == 0]
for col, lo, hi, nb in [("HbA1c_level", 3.5, 9.0, 22), ("blood_glucose_level", 80, 300, 22), ("age", 0, 80, 16)]:
    edges = np.linspace(lo, hi, nb + 1)
    hn, _ = np.histogram(neg[col], bins=edges, density=True)
    hp, _ = np.histogram(pos[col], bins=edges, density=True)
    write_dat(f"diabetes_hist_{col}", {"x": (edges[:-1] + edges[1:]) / 2, "neg": hn, "pos": hp})
corr_cols = DIAB_NUMERIC + ["hypertension", "heart_disease", DIAB_TARGET]
corr = df[corr_cols].corr().to_numpy()
write_matrix_dat("diabetes_corr", corr)
print(pd.DataFrame(corr, index=corr_cols, columns=corr_cols).round(2))



def pure_threshold(col):
    """Giá trị nhỏ nhất t sao cho mọi dòng có col >= t đều dương tính."""
    for v in np.sort(df[col].unique()):
        if df.loc[df[col] >= v, DIAB_TARGET].min() == 1:
            return float(v)


pure = {c: pure_threshold(c) for c in ("HbA1c_level", "blood_glucose_level")}
covered = ((df["HbA1c_level"] >= pure["HbA1c_level"]) | (df["blood_glucose_level"] >= pure["blood_glucose_level"]))
pos_covered = float(covered[df[DIAB_TARGET] == 1].mean())
print("Ngưỡng mà trên đó 100% dương tính:", pure, "· tỉ lệ ca dương nằm trên ít nhất một ngưỡng:", round(pos_covered, 4))

D = prepare_diabetes()
diab = {"n_raw": len(raw), "n_dup": dup, "n_rows": len(df), "n_features_raw": raw.shape[1] - 1,
        "n_features_encoded": len(D["feature_names"]), "pos_rate": df[DIAB_TARGET].mean(),
        "n_pos": int(df[DIAB_TARGET].sum()), "no_info_rate": (df["smoking_history"] == "No Info").mean(),
        "n_train": len(D["y"]["train"]), "n_val": len(D["y"]["val"]), "n_test": len(D["y"]["test"]),
        "desc": {c: desc.loc[c].to_dict() for c in DIAB_NUMERIC},
        "corr_target": {c: corr[corr_cols.index(c), -1] for c in corr_cols[:-1]},
        "corr_names": corr_cols, "pure_threshold": pure, "pos_covered": pos_covered,
        "categories": {c: sorted(df[c].unique().tolist()) for c in DIAB_CATEGORICAL}}
_ = save_metrics("data", {"cifar10": info10, "cifar100": info100, "diabetes": diab})

                      age   bmi  HbA1c_level  blood_glucose_level  \
age                  1.00  0.34         0.11                 0.11   
bmi                  0.34  1.00         0.08                 0.09   
HbA1c_level          0.11  0.08         1.00                 0.17   
blood_glucose_level  0.11  0.09         0.17                 1.00   
hypertension         0.26  0.15         0.08                 0.08   
heart_disease        0.24  0.06         0.07                 0.07   
diabetes             0.26  0.21         0.41                 0.42   

                     hypertension  heart_disease  diabetes  
age                          0.26           0.24      0.26  
bmi                          0.15           0.06      0.21  
HbA1c_level                  0.08           0.07      0.41  
blood_glucose_level          0.08           0.07      0.42  
hypertension                 1.00           0.12      0.20  
heart_disease                0.12           1.00      0.17  
diabetes            